In [1]:
!uv sync

Resolved 17 packages in 22ms
Audited 16 packages in 4ms


In [ ]:
from mandrop.engine import make_step, feq_fn, ex_jnp, ey_jnp
from mandrop.generator import setup

import jax
import jax.numpy as jnp
from jax import lax
import matplotlib.pyplot as plt

print(f"JAX {jax.__version__}, devices: {jax.devices()}")

In [ ]:
Nx, Ny = 200, 600

W = 3.0
sigma = 0.01
beta  = 3.0 * sigma / W
kappa = 6.0 * sigma * W

rho0 = 1.0
nu   = 1.0 / 6.0
tau_f = 3.0 * nu + 0.5
M_ch = 0.01
drho = 0.001

geo = setup(
    Nx=Nx, Ny=Ny,
    rho_in_water=rho0 + drho / 2.0,
    rho_in_oil=rho0 + drho / 2.0,
    rho_out=rho0 - drho / 2.0,
)
p = geo["params"]

print(f"Domain: {Nx}×{Ny}")
print(f"Main channel: w={p['w_channel']}, side={p['w_side']}")
print(f"Junction y={p['junction_y']} [{p['jy_bot']},{p['jy_top']}]")
print(f"Δρ={drho}, tau={tau_f}")

In [ ]:
wall = geo["wall"]

fig, ax = plt.subplots(figsize=(4, 12))
ax.imshow(wall.T, origin='lower', cmap='gray_r', vmin=0, vmax=1)
ax.set_title('Wall mask (black=wall, white=fluid)')
ax.axhline(p['jy_top'], color='r', ls=':', alpha=0.5, label='junction top')
ax.axhline(p['jy_bot'], color='b', ls=':', alpha=0.5, label='junction bot')
ax.legend(loc='upper right')
plt.show()

print(f"Fluid cells: {int((~wall).sum())}")
print(f"Wall cells: {int(wall.sum())}")

In [ ]:
interior = geo["interior"]

step = make_step(
    geo["wall"], geo["fluid"], geo["interior"], geo["opp_jnp"],
    tau_f, beta, kappa, M_ch,
    geo["apply_f_bcs"], geo["apply_phi_bcs"], geo["boundary_mask"],
)

print(f"Cross-junction flow-focusing: {Nx}×{Ny}")
print(f"Water inlet (top): x=[{p['x_L']+1},{p['x_R']-1}], rho={p['rho_in_water']:.4f}")
print(f"Oil inlets (sides): y=[{p['jy_bot']},{p['jy_top']-1}], rho={p['rho_in_oil']:.4f}")
print(f"Outlet (bottom): x=[{p['x_L']+1},{p['x_R']-1}], rho={p['rho_out']:.4f}")

In [ ]:
# === BC Verification ===
from mandrop.engine import zou_he_left, zou_he_right

xL, xR = p['x_L'] + 1, p['x_R']
jyb, jyt = p['jy_bot'], p['jy_top']
rho_oil = p['rho_in_oil']

# Test 1: apply BCs to equilibrium f, check rho at boundaries
f_test = feq_fn(jnp.ones((Nx, Ny)) * rho0, jnp.zeros((Nx, Ny)), jnp.zeros((Nx, Ny)))
f_test = geo["apply_f_bcs"](f_test)
rho_test = jnp.sum(f_test, axis=-1)

print("After apply_f_bcs on equilibrium:")
print(f"  LEFT  rho[-1,jyb:jyt]: [{float(rho_test[0, jyb:jyt].min()):.6f}, {float(rho_test[0, jyb:jyt].max()):.6f}]  (target={rho_oil})")
print(f"  RIGHT rho[-1,jyb:jyt]: [{float(rho_test[-1, jyb:jyt].min()):.6f}, {float(rho_test[-1, jyb:jyt].max()):.6f}]  (target={rho_oil})")
print(f"  TOP   rho[xL:xR,-1]:   [{float(rho_test[xL:xR, -1].min()):.6f}, {float(rho_test[xL:xR, -1].max()):.6f}]")
print(f"  BOT   rho[xL:xR, 0]:   [{float(rho_test[xL:xR, 0].min()):.6f}, {float(rho_test[xL:xR, 0].max()):.6f}]")

# Test 2: apply zou_he_right ALONE to equilibrium
f_test2 = feq_fn(jnp.ones((Nx, Ny)) * rho0, jnp.zeros((Nx, Ny)), jnp.zeros((Nx, Ny)))
f_test2 = zou_he_right(f_test2, jyb, jyt, rho_oil)
rho_test2 = jnp.sum(f_test2, axis=-1)
print(f"\nAfter zou_he_right ALONE on equilibrium:")
print(f"  RIGHT rho[-1,jyb:jyt]: [{float(rho_test2[-1, jyb:jyt].min()):.6f}, {float(rho_test2[-1, jyb:jyt].max()):.6f}]")

# Test 3: run 1 step, check rho
phi_test = geo["apply_phi_bcs"](jnp.ones((Nx, Ny)))
state_test = step((f_test, phi_test))
f_1, phi_1 = state_test
rho_1 = jnp.sum(f_1, axis=-1)
print(f"\nAfter 1 step:")
print(f"  LEFT  rho[0,jyb:jyt]:  [{float(rho_1[0, jyb:jyt].min()):.6f}, {float(rho_1[0, jyb:jyt].max()):.6f}]")
print(f"  RIGHT rho[-1,jyb:jyt]: [{float(rho_1[-1, jyb:jyt].min()):.6f}, {float(rho_1[-1, jyb:jyt].max()):.6f}]")

# Test 4: run 10 steps, check rho
state_t = (f_test, phi_test)
for _ in range(10):
    state_t = step(state_t)
f_10, _ = state_t
rho_10 = jnp.sum(f_10, axis=-1)
print(f"\nAfter 10 steps:")
print(f"  LEFT  rho[0,jyb:jyt]:  [{float(rho_10[0, jyb:jyt].min()):.6f}, {float(rho_10[0, jyb:jyt].max()):.6f}]")
print(f"  RIGHT rho[-1,jyb:jyt]: [{float(rho_10[-1, jyb:jyt].min()):.6f}, {float(rho_10[-1, jyb:jyt].max()):.6f}]")
print(f"  Global rho range: [{float(rho_10.min()):.6f}, {float(rho_10.max()):.6f}]")
print(f"  max|u|: {float(jnp.max(jnp.sqrt((jnp.sum(f_10*ex_jnp,axis=-1)/rho_10)**2 + (jnp.sum(f_10*ey_jnp,axis=-1)/rho_10)**2))):.4e}")

In [ ]:
apply_phi_bcs = geo["apply_phi_bcs"]

phi0 = jnp.ones((Nx, Ny))
rho_init = jnp.ones((Nx, Ny)) * rho0
f0 = feq_fn(rho_init, jnp.zeros((Nx, Ny)), jnp.zeros((Nx, Ny)))
phi0_box = apply_phi_bcs(phi0)

print(f"Flow-focusing: {Nx}×{Ny}, water(top) + oil(sides) → outlet(bottom)")
print(f"Δρ={drho}")

state = (f0, phi0_box)
state = step(state)
print("JIT compiled.")

state = (f0, phi0_box)
chunk_size = 500
n_chunks = 60  # 30k total steps

def scan_body(state, _):
    return step(state), None

import time
t0 = time.time()

xL, xR = p['x_L'] + 1, p['x_R']
jyb, jyt = p['jy_bot'], p['jy_top']

for chunk in range(n_chunks):
    state, _ = lax.scan(scan_body, state, None, length=chunk_size)
    f_c, phi_c = state
    f_c.block_until_ready()

    step_num = (chunk + 1) * chunk_size
    rho_c = jnp.sum(f_c, axis=-1)
    ux_c = jnp.sum(f_c * ex_jnp, axis=-1) / rho_c
    uy_c = jnp.sum(f_c * ey_jnp, axis=-1) / rho_c

    # Per-boundary stats
    # Water inlet (top): y=-1
    rho_top = rho_c[xL:xR, -1]
    uy_top = uy_c[xL:xR, -1]
    phi_top = phi_c[xL:xR, -1]

    # Oil inlet left: x=0
    rho_left = rho_c[0, jyb:jyt]
    ux_left = ux_c[0, jyb:jyt]
    phi_left = phi_c[0, jyb:jyt]

    # Oil inlet right: x=-1
    rho_right = rho_c[-1, jyb:jyt]
    ux_right = ux_c[-1, jyb:jyt]
    phi_right = phi_c[-1, jyb:jyt]

    # Outlet (bottom): y=0
    rho_bot = rho_c[xL:xR, 0]
    uy_bot = uy_c[xL:xR, 0]
    phi_bot = phi_c[xL:xR, 0]

    max_vel = jnp.max(jnp.sqrt(ux_c**2 + uy_c**2))
    n_water = ((phi_c < 0.5).astype(jnp.float64) * interior).sum()

    print(f"\n=== Step {step_num} === max|u|={float(max_vel):.2e}  water_px={float(n_water):.0f}  rho=[{float(rho_c.min()):.4f},{float(rho_c.max()):.4f}]")
    print(f"  TOP (water in):  rho=[{float(rho_top.min()):.4f},{float(rho_top.max()):.4f}]  uy=[{float(uy_top.min()):.4e},{float(uy_top.max()):.4e}]  phi=[{float(phi_top.min()):.3f},{float(phi_top.max()):.3f}]")
    print(f"  LEFT (oil in):   rho=[{float(rho_left.min()):.4f},{float(rho_left.max()):.4f}]  ux=[{float(ux_left.min()):.4e},{float(ux_left.max()):.4e}]  phi=[{float(phi_left.min()):.3f},{float(phi_left.max()):.3f}]")
    print(f"  RIGHT (oil in):  rho=[{float(rho_right.min()):.4f},{float(rho_right.max()):.4f}]  ux=[{float(ux_right.min()):.4e},{float(ux_right.max()):.4e}]  phi=[{float(phi_right.min()):.3f},{float(phi_right.max()):.3f}]")
    print(f"  BOT (outlet):    rho=[{float(rho_bot.min()):.4f},{float(rho_bot.max()):.4f}]  uy=[{float(uy_bot.min()):.4e},{float(uy_bot.max()):.4e}]  phi=[{float(phi_bot.min()):.3f},{float(phi_bot.max()):.3f}]")

    if jnp.isnan(phi_c).any() or jnp.isnan(f_c).any():
        print("  *** NaN detected! ***")
        break

t1 = time.time()
f_final, phi_final = state
print(f"\nDone in {t1-t0:.1f}s ({n_chunks*chunk_size/(t1-t0):.0f} steps/s)")

In [ ]:
rho_final = jnp.sum(f_final, axis=-1)
ux_final = jnp.sum(f_final * ex_jnp, axis=-1) / rho_final
uy_final = jnp.sum(f_final * ey_jnp, axis=-1) / rho_final
vel_mag = jnp.sqrt(ux_final**2 + uy_final**2)

n_water = ((phi_final < 0.5).astype(jnp.float64) * interior).sum()

print("=" * 50)
print(f"FLOW-FOCUSING — {n_chunks*chunk_size} STEPS")
print("=" * 50)
print(f"Max velocity:       {vel_mag.max():.4e} lu")
print(f"Water pixels:       {n_water:.0f}")
print(f"phi range:          [{phi_final.min():.6f}, {phi_final.max():.6f}]")
print(f"rho range:          [{rho_final.min():.6f}, {rho_final.max():.6f}]")

In [ ]:
vel_mag = jnp.sqrt(ux_final**2 + uy_final**2)

fig, axes = plt.subplots(1, 4, figsize=(16, 12))

im0 = axes[0].imshow(phi_final.T, origin='lower', cmap='RdBu', vmin=0, vmax=1)
axes[0].set_title('φ (oil=1, water=0)')
plt.colorbar(im0, ax=axes[0], shrink=0.5)

im1 = axes[1].imshow(rho_final.T, origin='lower', cmap='viridis')
axes[1].set_title('ρ (pressure)')
plt.colorbar(im1, ax=axes[1], shrink=0.5)

im2 = axes[2].imshow(uy_final.T, origin='lower', cmap='coolwarm',
                       vmin=-vel_mag.max(), vmax=vel_mag.max())
axes[2].set_title('u_y (flow direction)')
plt.colorbar(im2, ax=axes[2], shrink=0.5)

im3 = axes[3].imshow(vel_mag.T, origin='lower', cmap='hot')
axes[3].set_title(f'|u| (max={vel_mag.max():.2e})')
plt.colorbar(im3, ax=axes[3], shrink=0.5)

for ax in axes:
    ax.set_aspect('equal')

plt.tight_layout()
plt.show()